In [518]:
from lxml import html
from bs4 import BeautifulSoup
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import aiohttp
import asyncio
import requests
import random
import time
import json
import re

In [519]:
def get_html(session, url):
    while True:
        try:
            response = session.get(url, timeout=30)

            if response.status_code == 429:
                print("429 received. Waiting 5 minutes...")
                time.sleep(300)
                continue

            response.raise_for_status()
            return response

        except requests.exceptions.RequestException as e:
            print(f"{type(e).__name__}: {e}")
            print("Retrying in 30 seconds...")
            time.sleep(30)

In [520]:
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
})

with open("link.json") as file:
    data = json.load(file)

In [521]:
data

{'website': 'https://www.gsmarena.com/'}

In [522]:
url = data["website"]
url

'https://www.gsmarena.com/'

In [523]:
filename = "top_page.html"
directory = Path("cache")
CACHE_FILE = directory

In [524]:
if CACHE_FILE.exists():
    print(f"Loading {url} from cache...")
    with open(CACHE_FILE / filename, "r", encoding="utf-8") as file:
        html_content = file.read()
else:
    CACHE_FILE.mkdir(parents=True, exist_ok=True)

    print(f"Downloading {url}...")
    
    html_content = get_html(session, url)

    if html_content.status_code == 429:
        print(f"HTTP Status {html_content.status_code} received || Sleeping for 5 minutes before retrying || url: {url}...")
        time.sleep(300)
        html_content = get_html(session, url)

    html_content = html_content.content

    with open(CACHE_FILE / filename, "wb") as file:
        file.write(html_content)

In [525]:
tree = html.fromstring(html_content)
page_brand = tree.xpath('//div[@class="brandmenu-v2 light l-box clearfix"]/ul/li/a')

In [526]:
for i in page_brand:
    brand_name = i.text
    brand_link = i.get("href")
    print(f"Link: {brand_link} \t Brand: {brand_name}")

Link: samsung-phones-9.php 	 Brand: Samsung
Link: apple-phones-48.php 	 Brand: Apple
Link: huawei-phones-58.php 	 Brand: Huawei
Link: nokia-phones-1.php 	 Brand: Nokia
Link: sony-phones-7.php 	 Brand: Sony
Link: lg-phones-20.php 	 Brand: LG
Link: htc-phones-45.php 	 Brand: HTC
Link: motorola-phones-4.php 	 Brand: Motorola
Link: lenovo-phones-73.php 	 Brand: Lenovo
Link: xiaomi-phones-80.php 	 Brand: Xiaomi
Link: google-phones-107.php 	 Brand: Google
Link: honor-phones-121.php 	 Brand: Honor
Link: oppo-phones-82.php 	 Brand: Oppo
Link: realme-phones-118.php 	 Brand: Realme
Link: oneplus-phones-95.php 	 Brand: OnePlus
Link: nothing-phones-128.php 	 Brand: Nothing
Link: vivo-phones-98.php 	 Brand: vivo
Link: meizu-phones-74.php 	 Brand: Meizu
Link: ulefone_-phones-124.php 	 Brand: Ulefone
Link: alcatel-phones-5.php 	 Brand: Alcatel
Link: zte-phones-62.php 	 Brand: ZTE
Link: rugone-phones-136.php 	 Brand: RugOne
Link: umidigi-phones-135.php 	 Brand: Umidigi
Link: coolpad-phones-105.php 	 B

In [527]:
print(f"Link: {url}{page_brand[0].get('href')} \t Brand: {page_brand[0].text}")

Link: https://www.gsmarena.com/samsung-phones-9.php 	 Brand: Samsung


Download Top Page (First page for each brand)

In [528]:
brand_urls = []
new_url = url + page_brand[0].get('href')

pbar = tqdm(page_brand, ascii=True)
pbar.set_description("Downloading brand pages...")

for i, b_url in enumerate(pbar):
    
    new_url = url + b_url.get('href')

    # cache the top brand page
    filename = f"top_{b_url.text}_page.html"
    directory = Path("cache/pages") / b_url.text
    CACHE_FILE = directory

    if CACHE_FILE.exists():
        pbar.set_description(f"HTML is already cached for {new_url}...")
        with open(CACHE_FILE / filename, "r", encoding="utf-8") as file:
            html_content = file.read()
    else:
        # sleep time between 2.5 to 3.5 seconds to avoid being blocked by the server
        sleep_time = random.uniform(2.5, 3.5)
        time.sleep(sleep_time)

        pbar.set_description(f"Creating {b_url.text} Directory || Downloading {new_url} || sleeping for {sleep_time:.2f} seconds...")

        CACHE_FILE.mkdir(parents=True, exist_ok=True)

        html_content = get_html(session, new_url)
        
        if html_content.status_code == 429:
                pbar.set_description(f"HTTP Status {html_content.status_code} received || Sleeping for 5 minutes before retrying || url: {new_url}...")
                time.sleep(300)
                html_content = get_html(session, new_url)

        html_content = html_content.content

        with open(CACHE_FILE / filename, "wb") as file:
            file.write(html_content)

    # fetch the number of pages for each brand
    tree = html.fromstring(html_content)
    brand_page_count = tree.xpath('//div[@class="nav-pages"]/a')

    paged_url = f"{url}{re.sub(r"p\d", "p{}", brand_page_count[1].get('href'))}" if len(brand_page_count) > 0 else None
    max_page = brand_page_count[-2].text if len(brand_page_count) > 0 else 0

    # json information for each brand 
    brand_urls.append({
        "brand": b_url.text,
        "top_url": new_url,
        "paged_url": paged_url,
        "max_page": int(max_page),
        "cache": str(Path("cache") / b_url.text)
    })  

Creating TCL Directory || Downloading https://www.gsmarena.com/tcl-phones-123.php || sleeping for 3.35 seconds...: 100%|##########| 36/36 [02:26<00:00,  4.06s/it]            


Saving json information about brands, max pages, urls

In [529]:
directory = Path("metadata")
directory.mkdir(parents=True, exist_ok=True)

CACHE_FILE = directory / "brands_urls_information.json"

if CACHE_FILE.exists():
    print(f"metadata file already exists.")
else:
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(brand_urls, f, indent=4)

JSON Check

In [530]:
brand_urls

[{'brand': 'Samsung',
  'top_url': 'https://www.gsmarena.com/samsung-phones-9.php',
  'paged_url': 'https://www.gsmarena.com/samsung-phones-f-9-0-p{}.php',
  'max_page': 29,
  'cache': 'cache/Samsung'},
 {'brand': 'Apple',
  'top_url': 'https://www.gsmarena.com/apple-phones-48.php',
  'paged_url': 'https://www.gsmarena.com/apple-phones-f-48-0-p{}.php',
  'max_page': 3,
  'cache': 'cache/Apple'},
 {'brand': 'Huawei',
  'top_url': 'https://www.gsmarena.com/huawei-phones-58.php',
  'paged_url': 'https://www.gsmarena.com/huawei-phones-f-58-0-p{}.php',
  'max_page': 11,
  'cache': 'cache/Huawei'},
 {'brand': 'Nokia',
  'top_url': 'https://www.gsmarena.com/nokia-phones-1.php',
  'paged_url': 'https://www.gsmarena.com/nokia-phones-f-1-0-p{}.php',
  'max_page': 12,
  'cache': 'cache/Nokia'},
 {'brand': 'Sony',
  'top_url': 'https://www.gsmarena.com/sony-phones-7.php',
  'paged_url': 'https://www.gsmarena.com/sony-phones-f-7-0-p{}.php',
  'max_page': 4,
  'cache': 'cache/Sony'},
 {'brand': 'LG'

In [531]:
with open("metadata/brands_urls_information.json", "r", encoding="utf-8") as f:
    brand_urls = json.load(f)

In [532]:
brand_urls[1]

{'brand': 'Apple',
 'top_url': 'https://www.gsmarena.com/apple-phones-48.php',
 'paged_url': 'https://www.gsmarena.com/apple-phones-f-48-0-p{}.php',
 'max_page': 3,
 'cache': 'cache/Apple'}

Download Paginated Page 

In [533]:
pbar = tqdm(brand_urls, ascii=True)
pbar.set_description("Downloading brand paginated pages...")

for i, brand in enumerate(pbar):
    pbar.set_description(f" brand  {brand['brand']} || max_page: {brand['max_page']}")
    
    for j in range(1, brand['max_page'] + 1): # Download from page 1 to max_page
        
        new_url = brand['paged_url'].format(j)  # Format the URL with page number j

        pbar.set_description(f"{brand['brand']} Page {j} / {brand['max_page']} || url: {new_url} || slept for {sleep_time:.2f} seconds...")

        directory = Path("cache/pages") / brand['brand']
        filename = f"{brand['brand']}_page_{j}.html"
        CACHE_FILE = directory / filename

        if CACHE_FILE.exists():
            pbar.set_description(f"Paged HTML is already cached for {new_url}...")
        else:
            # sleep time between 2.5 to 3.5 seconds to avoid being blocked by the server
            sleep_time = random.uniform(2.5, 3.5)
            time.sleep(sleep_time)

            
            html_content = get_html(session, new_url)
            
            if html_content.status_code == 429:
                pbar.set_description(f"HTTP Status {html_content.status_code} received || Sleeping for 5 minutes before retrying || url: {new_url}...")
                time.sleep(300)
                html_content = get_html(session, new_url)
                
            html_content = html_content.content

            if j != 1:  # Skip saving the first page since it's already saved as top_url
                with open(CACHE_FILE, "wb") as file:
                    file.write(html_content)

    

TCL Page 2 / 2 || url: https://www.gsmarena.com/tcl-phones-f-123-0-p2.php || slept for 3.37 seconds...: 100%|##########| 36/36 [14:36<00:00, 24.36s/it]            


Saving individual phone urls into json

In [534]:
phone_urls = []

pbr = tqdm(brand_urls, ascii=True)
for brand in pbr:
    directory = Path("cache/pages") / brand['brand']
    CACHE_FILE = directory

    for html_file in directory.rglob("*.html"):

        html_content = html_file.read_text(encoding="utf-8")

        # parse with lxml
        tree = html.fromstring(html_content)
        data = tree.xpath('//div[@class="makers"]/ul/li/a')

        for link in data:
            pbr.set_description(f"Parsing file: {html_file.name} || url: {link.get('href')}")
            phone_urls.append({
                "brand": brand['brand'],
                "url": f"{url}{link.get('href')}"
            })
        

Parsing file: TCL_page_2.html || url: tcl_plex-9841.php: 100%|##########| 36/36 [00:06<00:00,  5.77it/s]                                    


Saving individuals phone url json

In [535]:
directory = Path("metadata")
directory.mkdir(parents=True, exist_ok=True)

CACHE_FILE = directory / "phone_urls.json"

if CACHE_FILE.exists():
    print(f"metadata file already exists.")
else:
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(phone_urls, f, indent=4)

Downloading individuals html for the phone specs

In [536]:
directory = Path("cache/phones")
directory.mkdir(parents=True, exist_ok=True)

CACHE_FILE = directory / "phone_urls.json"

Testing

In [537]:
len(phone_urls)

9996

In [538]:
from collections import Counter

Counter(phone["brand"] for phone in phone_urls)

Counter({'Samsung': 1450,
         'Motorola': 700,
         'LG': 667,
         'vivo': 600,
         'Nokia': 592,
         'Huawei': 532,
         'Xiaomi': 525,
         'ZTE': 437,
         'Oppo': 419,
         'Alcatel': 414,
         'Honor': 329,
         'HTC': 297,
         'Realme': 294,
         'Micromax': 289,
         'Lenovo': 263,
         'Asus': 207,
         'Tecno': 191,
         'Infinix': 170,
         'Sony': 163,
         'Doogee': 162,
         'Ulefone': 148,
         'Apple': 146,
         'Oukitel': 117,
         'Blackview': 114,
         'OnePlus': 113,
         'Cubot': 100,
         'TCL': 100,
         'Meizu': 87,
         'Sharp': 83,
         'Umidigi': 80,
         'Itel': 62,
         'Coolpad': 56,
         'Google': 40,
         'Oscal': 31,
         'Nothing': 15,
         'RugOne': 3})

In [539]:
counter = 0
list_to_check = []
for phone in phone_urls:
    if re.search(r".*meizu.*", phone["url"]) and phone["brand"] == "Meizu":
        counter += 1
        list_to_check.append(phone['url'])
        print(f"Valid URL {counter}: {phone['url']}")

set_to_check = set(list_to_check)
print(f"Number of unique Meizu URLs: {len(set_to_check)}")

Valid URL 1: https://www.gsmarena.com/meizu_22_5g-14150.php
Valid URL 2: https://www.gsmarena.com/meizu_note_22_pro_5g-13884.php
Valid URL 3: https://www.gsmarena.com/meizu_note_22_5g-13883.php
Valid URL 4: https://www.gsmarena.com/meizu_mblu_22_pro-13700.php
Valid URL 5: https://www.gsmarena.com/meizu_mblu_22-13699.php
Valid URL 6: https://www.gsmarena.com/meizu_note_22-13698.php
Valid URL 7: https://www.gsmarena.com/meizu_note_16_5g-13862.php
Valid URL 8: https://www.gsmarena.com/meizu_note_16_pro_5g-13861.php
Valid URL 9: https://www.gsmarena.com/meizu_mblu_21-13530.php
Valid URL 10: https://www.gsmarena.com/meizu_lucky_08-13365.php
Valid URL 11: https://www.gsmarena.com/meizu_note_21_pro-13341.php
Valid URL 12: https://www.gsmarena.com/meizu_note_21-13342.php
Valid URL 13: https://www.gsmarena.com/meizu_blue_20-13212.php
Valid URL 14: https://www.gsmarena.com/meizu_21_note-13007.php
Valid URL 15: https://www.gsmarena.com/meizu_21_pro-12805.php
Valid URL 16: https://www.gsmarena.com